# 09 — Graph Visualizations for Presentation

Generate network visualizations for each graph level. Nodes are colored by drug category, sized by degree.


In [ ]:
import pandas as pd
import numpy as np
import re
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 150


In [ ]:
# Drug categories
LIQUID_DRUGS = {
    "Imatinib", "Dasatinib", "Nilotinib", "Bosutinib", "Ponatinib",
    "Daunorubicin", "Idarubicin", "Cytarabine", "Mitoxantrone",
    "Ibrutinib", "Idelalisib", "Venetoclax",
    "Bortezomib", "Carfilzomib", "Ixazomib",
    "Thalidomide", "Lenalidomide", "Pomalidomide",
    "Vincristine", "Vinblastine", "Cyclophosphamide", "Chlorambucil",
    "Mechlorethamine", "Melphalan", "Busulfan",
    "Bleomycin", "Etoposide", "Procarbazine",
    "Ruxolitinib", "Baricitinib", "Tofacitinib",
    "Methotrexate", "Carmustine", "Lomustine", "Fludarabine",
}

SOLID_DRUGS = {
    "Tamoxifen", "Fulvestrant", "Anastrozole", "Letrozole", "Exemestane",
    "Lapatinib", "Neratinib", "Tucatinib",
    "Palbociclib", "Ribociclib", "Abemaciclib",
    "Paclitaxel", "Docetaxel", "Doxorubicin", "Capecitabine",
    "Erlotinib", "Gefitinib", "Afatinib", "Osimertinib", "Icotinib",
    "Crizotinib", "Ceritinib", "Alectinib", "Brigatinib",
    "Vemurafenib", "Dabrafenib", "Encorafenib",
    "Fluorouracil", "Irinotecan", "Oxaliplatin", "Regorafenib",
    "Sorafenib", "Sunitinib", "Pazopanib", "Axitinib", "Cabozantinib", "Lenvatinib",
    "Everolimus", "Temsirolimus",
    "Cisplatin", "Carboplatin", "Olaparib", "Rucaparib", "Niraparib",
    "Bicalutamide", "Flutamide", "Enzalutamide", "Abiraterone",
    "Diethylstilbestrol", "Degarelix", "Goserelin", "Leuprolide",
    "Gemcitabine", "Temozolomide", "Dacarbazine", "Topotecan",
    "Ifosfamide", "Altretamine", "Estramustine", "Thiotepa",
    "Mitomycin", "Teniposide", "Vinorelbine", "Vandetanib", "Epirubicin",
}

ALL_ONCOLOGY = LIQUID_DRUGS | SOLID_DRUGS

# NLP patterns
PATTERNS = [
    (r"metabolism.*(?:increased|decreased)", "metabolism"),
    (r"active metabolites", "metabolism"),
    (r"serum concentration.*(?:increased|decreased|reduced)", "concentration"),
    (r"bioavailability.*(?:increased|decreased)", "concentration"),
    (r"protein binding", "concentration"),
    (r"risk or severity of adverse effects", "adverse_effects"),
    (r"risk or severity of.*(?:prolongation|bleeding|hypotension|hypertension|heart failure|hyperkalemia)", "adverse_effects"),
    (r"risk of a hypersensitivity", "adverse_effects"),
    (r"therapeutic efficacy.*(?:decreased|increased)", "efficacy"),
    (r"decrease effectiveness", "efficacy"),
    (r"decrease in the absorption", "absorption"),
    (r"absorption.*(?:increased|decreased)", "absorption"),
    (r"excretion rate", "excretion"),
    (r"(?:increase|decrease).*(?:activities|activity)", "activity"),
]

def extract_label(desc):
    for p, l in PATTERNS:
        if re.search(p, desc.lower()):
            return l
    return None

# Load data
df_full = pd.read_csv("../data/drug_interactions.csv")
df_full['label'] = df_full['Interaction Description'].apply(extract_label)
df_full = df_full.dropna(subset=['label'])

all_drugs = set(df_full['Drug 1'].unique()) | set(df_full['Drug 2'].unique())
liquid_found = LIQUID_DRUGS & all_drugs
solid_found = SOLID_DRUGS & all_drugs
onc_found = ALL_ONCOLOGY & all_drugs

print(f"Liquid drugs in dataset: {len(liquid_found)}")
print(f"Solid drugs in dataset: {len(solid_found)}")
print(f"Total oncology: {len(onc_found)}")


In [ ]:
def get_node_color(drug):
    """Assign color based on drug category."""
    if drug in liquid_found:
        return '#e74c3c'  # Red for liquid
    elif drug in solid_found:
        return '#f39c12'  # Orange for solid
    else:
        return '#3498db'  # Blue for non-oncology

def draw_graph(G, title, filename, max_nodes=300, seed=42, figsize=(12, 12)):
    """
    Draw a network graph with:
    - Nodes colored by drug category (liquid/solid/non-oncology)
    - Node size proportional to degree
    - Edges colored by interaction type
    """
    np.random.seed(seed)
    
    # Sample nodes if graph is too large
    if G.number_of_nodes() > max_nodes:
        # Keep all oncology nodes, sample the rest
        onc_nodes = [n for n in G.nodes() if n in onc_found]
        non_onc_nodes = [n for n in G.nodes() if n not in onc_found]
        
        # Sample non-oncology nodes
        n_sample = max_nodes - len(onc_nodes)
        if n_sample > 0 and len(non_onc_nodes) > n_sample:
            # Prefer high-degree non-oncology nodes
            non_onc_degs = {n: G.degree(n) for n in non_onc_nodes}
            sorted_non_onc = sorted(non_onc_degs, key=non_onc_degs.get, reverse=True)
            sampled_non_onc = sorted_non_onc[:n_sample]
        else:
            sampled_non_onc = non_onc_nodes
        
        keep_nodes = set(onc_nodes) | set(sampled_non_onc)
        G = G.subgraph(keep_nodes).copy()
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Layout
    pos = nx.spring_layout(G, k=1.5/np.sqrt(G.number_of_nodes()), 
                           iterations=50, seed=seed)
    
    # Node colors and sizes
    node_colors = [get_node_color(n) for n in G.nodes()]
    degrees = dict(G.degree())
    max_deg = max(degrees.values()) if degrees else 1
    node_sizes = [max(20, 300 * (degrees[n] / max_deg)) for n in G.nodes()]
    
    # Edge colors by interaction type
    edge_color_map = {
        'metabolism': '#2ecc71',
        'concentration': '#3498db',
        'adverse_effects': '#e74c3c',
        'activity': '#f39c12',
        'efficacy': '#9b59b6',
        'excretion': '#1abc9c',
        'absorption': '#e67e22',
    }
    edge_colors = []
    for u, v, data in G.edges(data=True):
        label = data.get('label', '')
        edge_colors.append(edge_color_map.get(label, '#cccccc'))
    
    # Draw edges first (behind nodes)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.08, 
                           edge_color=edge_colors, width=0.3,
                           arrows=False)
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                           node_size=node_sizes, alpha=0.7, 
                           edgecolors='white', linewidths=0.3)
    
    # Label only high-degree oncology nodes
    labels = {}
    for n in G.nodes():
        if n in onc_found and degrees.get(n, 0) > np.percentile(list(degrees.values()), 90):
            labels[n] = n
    
    nx.draw_networkx_labels(G, pos, labels=labels, ax=ax,
                           font_size=6, font_weight='bold',
                           font_color='black')
    
    # Title
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    
    # Legend for node types
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c',
               markersize=10, label=f'Liquid Cancer ({sum(1 for n in G.nodes() if n in liquid_found)})'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#f39c12',
               markersize=10, label=f'Solid Cancer ({sum(1 for n in G.nodes() if n in solid_found)})'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db',
               markersize=10, label=f'Non-oncology ({sum(1 for n in G.nodes() if n not in onc_found)})'),
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=10,
             framealpha=0.9, edgecolor='gray')
    
    # Stats annotation
    stats_text = f"Nodes: {G.number_of_nodes():,}  |  Edges: {G.number_of_edges():,}"
    ax.annotate(stats_text, xy=(0.5, 0.02), xycoords='axes fraction',
               ha='center', fontsize=10, color='gray',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(f"../data/{filename}", dpi=200, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()
    print(f"Saved: {filename}")


## 1. Build Graphs

In [ ]:
# Build NetworkX graphs
def build_graph(df):
    G = nx.DiGraph()
    for _, row in df.iterrows():
        G.add_edge(row['Drug 1'], row['Drug 2'], label=row['label'])
    return G

df_onc = df_full[(df_full['Drug 1'].isin(onc_found)) | (df_full['Drug 2'].isin(onc_found))]
df_liquid = df_full[(df_full['Drug 1'].isin(liquid_found)) | (df_full['Drug 2'].isin(liquid_found))]
df_solid = df_full[(df_full['Drug 1'].isin(solid_found)) | (df_full['Drug 2'].isin(solid_found))]

G_full = build_graph(df_full)
G_onc = build_graph(df_onc)
G_liquid = build_graph(df_liquid)
G_solid = build_graph(df_solid)

for name, G in [("Full", G_full), ("Oncology", G_onc), ("Liquid", G_liquid), ("Solid", G_solid)]:
    print(f"{name:>10}: {G.number_of_nodes():>5} nodes, {G.number_of_edges():>7} edges")


## 2. Full Graph (Sampled)

In [ ]:
draw_graph(G_full, 
           "Full DDI Network (Top 300 Drugs by Degree)",
           "viz_graph_full.png",
           max_nodes=300, figsize=(14, 14))


## 3. Oncology Subgraph

In [ ]:
draw_graph(G_onc,
           "Oncology Subgraph (≥1 Oncology Drug per Edge)",
           "viz_graph_oncology.png",
           max_nodes=250, figsize=(14, 14))


## 4. Liquid Cancer Subgraph

In [ ]:
draw_graph(G_liquid,
           "Liquid Cancer (Hematological) Subgraph",
           "viz_graph_liquid.png",
           max_nodes=200, figsize=(14, 14))


## 5. Solid Cancer Subgraph

In [ ]:
draw_graph(G_solid,
           "Solid Tumor Subgraph",
           "viz_graph_solid.png",
           max_nodes=200, figsize=(14, 14))


## 6. Four-Way Overview (Presentation Slide)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 20))

graph_configs = [
    (G_full, "Full Graph", 200),
    (G_onc, "Oncology Subgraph", 200),
    (G_liquid, "Liquid Cancer", 150),
    (G_solid, "Solid Cancer", 150),
]

for ax, (G, title, max_n) in zip(axes.flat, graph_configs):
    np.random.seed(42)
    
    # Sample
    G_draw = G
    if G.number_of_nodes() > max_n:
        onc_nodes = [n for n in G.nodes() if n in onc_found]
        non_onc_nodes = [n for n in G.nodes() if n not in onc_found]
        non_onc_degs = {n: G.degree(n) for n in non_onc_nodes}
        sorted_non_onc = sorted(non_onc_degs, key=non_onc_degs.get, reverse=True)
        n_sample = max_n - len(onc_nodes)
        sampled = sorted_non_onc[:max(0, n_sample)]
        keep = set(onc_nodes) | set(sampled)
        G_draw = G.subgraph(keep).copy()
    
    pos = nx.spring_layout(G_draw, k=1.8/np.sqrt(G_draw.number_of_nodes()),
                           iterations=50, seed=42)
    
    node_colors = [get_node_color(n) for n in G_draw.nodes()]
    degrees = dict(G_draw.degree())
    max_deg = max(degrees.values()) if degrees else 1
    node_sizes = [max(10, 200 * (degrees[n] / max_deg)) for n in G_draw.nodes()]
    
    nx.draw_networkx_edges(G_draw, pos, ax=ax, alpha=0.05, width=0.2, arrows=False)
    nx.draw_networkx_nodes(G_draw, pos, ax=ax, node_color=node_colors,
                           node_size=node_sizes, alpha=0.7,
                           edgecolors='white', linewidths=0.2)
    
    ax.set_title(f"{title}\n{G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges",
                fontsize=13, fontweight='bold')
    ax.set_axis_off()

# Shared legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=12, label='Liquid Cancer'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f39c12', markersize=12, label='Solid Cancer'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=12, label='Non-oncology'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=12,
          framealpha=0.9, bbox_to_anchor=(0.5, -0.02))

plt.suptitle("DDI Network: Graph Hierarchy", fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("../data/viz_graph_4way_overview.png", dpi=200, bbox_inches='tight',
            facecolor='white')
plt.show()
print("Saved: viz_graph_4way_overview.png")
